# CONUS wildfire activity map

This notebook builds a Watch Duty-style **data view** for the contiguous United States (CONUS).

- **Small colored dots:** recent NASA FIRMS satellite heat detections
- **Dark red shapes:** current NIFC/WFIGS wildfire perimeters
- **VIIRS and MODIS:** separate layers that can be switched on and off

Satellite heat detections are clues, not confirmed fire boundaries. The perimeter layer is the official mapped boundary, but it can update more slowly.

> This is an independent map made from public data. It is not affiliated with Watch Duty.

In [ ]:
# Run this once in a new Jupyter environment.
%pip install -q pandas geopandas requests folium shapely pyogrio

## 1. Settings and private API keys

The notebook asks for your keys without printing or saving them in the notebook.

- Get a free NASA FIRMS map key: https://firms.modaps.eosdis.nasa.gov/api/map_key/
- Enter your CARTO key to use CARTO Voyager. Leave it blank to use OpenStreetMap instead.

`DAYS = 1` means “show detections from the latest day.” The FIRMS Area API accepts 1–5 days.

In [ ]:
from datetime import datetime, timezone
from getpass import getpass
from html import escape
from io import StringIO
from pathlib import Path
import os
import time

import folium
import geopandas as gpd
import pandas as pd
import requests
from IPython.display import clear_output, display
from shapely.geometry import box

FIRMS_MAP_KEY = os.environ.get("FIRMS_MAP_KEY") or getpass("Enter your NASA FIRMS map key: ").strip()
CARTO_API_KEY = os.environ.get("CARTO_API_KEY") or getpass("Enter your CARTO API key (optional): ").strip()

if not FIRMS_MAP_KEY:
    raise ValueError("A NASA FIRMS map key is required.")

# west, south, east, north — approximate CONUS limits
CONUS_BBOX = (-125.0, 24.0, -66.5, 50.0)
CONUS_BOUNDS = [[24.0, -125.0], [50.0, -66.5]]
DAYS = 1
OUTPUT_HTML = Path("outputs/conus/conus_wildfire_map.html")
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

# Smaller markers and a clearer, color-blind-friendly palette.
# Set show=False on a source if you want it available but hidden at startup.
FIRMS_SOURCES = {
    "VIIRS_NOAA21_NRT": {
        "label": "VIIRS NOAA-21",
        "color": "#D55E00",
        "radius": 1.8,
        "show": True,
    },
    "VIIRS_NOAA20_NRT": {
        "label": "VIIRS NOAA-20",
        "color": "#E69F00",
        "radius": 1.8,
        "show": False,
    },
    "VIIRS_SNPP_NRT": {
        "label": "VIIRS Suomi-NPP",
        "color": "#CC79A7",
        "radius": 1.8,
        "show": False,
    },
    "MODIS_NRT": {
        "label": "MODIS",
        "color": "#0072B2",
        "radius": 2.2,
        "show": True,
    },
}

## 2. Download recent NASA FIRMS detections

Each sensor is downloaded separately. That lets the layer control show or hide VIIRS and MODIS independently.

In [ ]:
def download_firms_source(source, days=DAYS):
    bbox_text = ",".join(str(value) for value in CONUS_BBOX)
    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{FIRMS_MAP_KEY}/{source}/{bbox_text}/{days}"
    )

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    # FIRMS may return an empty response when a sensor has no detections.
    if not response.text.strip():
        return gpd.GeoDataFrame(
            columns=["latitude", "longitude", "source_sensor", "geometry"],
            geometry="geometry",
            crs="EPSG:4326",
        )

    frame = pd.read_csv(StringIO(response.text))
    if frame.empty:
        frame["source_sensor"] = source
        return gpd.GeoDataFrame(
            frame,
            geometry=gpd.GeoSeries([], crs="EPSG:4326"),
            crs="EPSG:4326",
        )

    required = {"latitude", "longitude"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"FIRMS response for {source} is missing: {sorted(missing)}")

    frame["source_sensor"] = source

    # Keep only points inside the CONUS map rectangle.
    west, south, east, north = CONUS_BBOX
    frame = frame[
        frame["longitude"].between(west, east)
        & frame["latitude"].between(south, north)
    ].copy()

    return gpd.GeoDataFrame(
        frame,
        geometry=gpd.points_from_xy(frame["longitude"], frame["latitude"]),
        crs="EPSG:4326",
    )


def download_all_firms():
    result = {}
    for source, style in FIRMS_SOURCES.items():
        detections = download_firms_source(source)
        result[source] = detections
        print(f"{style['label']}: {len(detections):,} detections")
    return result


firms_by_source = download_all_firms()

## 3. Download current NIFC/WFIGS wildfire perimeters

The query asks only for wildfire polygons that intersect the CONUS rectangle. It also handles pagination so the map is not silently limited to the first server page.

In [ ]:
WFIGS_URL = (
    "https://services3.arcgis.com/"
    "T4QMspbfLg3qTGWY/arcgis/rest/services/"
    "WFIGS_Interagency_Perimeters_Current/FeatureServer/0/query"
)

WFIGS_FIELDS = [
    "poly_IncidentName",
    "poly_GISAcres",
    "poly_DateCurrent",
    "poly_PolygonDateTime",
    "attr_IncidentName",
    "attr_IncidentSize",
    "attr_PercentContained",
    "attr_FireDiscoveryDateTime",
    "attr_ModifiedOnDateTime_dt",
    "attr_POOState",
    "attr_IrwinID",
]


def download_current_perimeters(page_size=2000):
    west, south, east, north = CONUS_BBOX
    all_features = []
    offset = 0

    while True:
        params = {
            "where": "attr_IncidentTypeCategory = 'WF'",
            "outFields": ",".join(WFIGS_FIELDS),
            "returnGeometry": "true",
            "geometry": f"{west},{south},{east},{north}",
            "geometryType": "esriGeometryEnvelope",
            "inSR": "4326",
            "spatialRel": "esriSpatialRelIntersects",
            "outSR": "4326",
            "resultOffset": offset,
            "resultRecordCount": page_size,
            "f": "geojson",
        }

        response = requests.get(WFIGS_URL, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            raise RuntimeError(f"WFIGS returned an error: {payload['error']}")

        page = payload.get("features", [])
        all_features.extend(page)

        if len(page) < page_size:
            break
        offset += page_size

    if not all_features:
        return gpd.GeoDataFrame(
            columns=WFIGS_FIELDS + ["geometry"],
            geometry="geometry",
            crs="EPSG:4326",
        )

    perimeters = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")

    # Clip anything extending beyond the requested display rectangle.
    # The server already restricted the request to CONUS.
    perimeters = perimeters[
        perimeters.geometry.notna()
        & ~perimeters.geometry.is_empty
    ].copy()

    return perimeters


perimeters = download_current_perimeters()
print(f"Current CONUS perimeters: {len(perimeters):,}")
perimeters.head()

## 4. Build the map

The map opens fitted to CONUS. The dots are intentionally small:

- VIIRS radius: **1.8 pixels**
- MODIS radius: **2.2 pixels** because MODIS detections represent a coarser footprint
- Perimeters: dark red outline with a light transparent fill

Use the layer control in the upper-right corner to compare sensors.

In [ ]:
def clean_value(value, default="N/A"):
    if value is None or pd.isna(value):
        return default
    return escape(str(value))


def build_fire_map(firms_layers, perimeter_data):
    fire_map = folium.Map(
        location=[39.5, -98.5],
        zoom_start=4,
        min_zoom=4,
        max_zoom=13,
        tiles=None,
        control_scale=True,
        prefer_canvas=True,
    )

    if CARTO_API_KEY:
        tile_url = (
            "https://basemaps.cartocdn.com/rastertiles/"
            f"voyager/{{z}}/{{x}}/{{y}}.png?key={CARTO_API_KEY}"
        )
        folium.TileLayer(
            tiles=tile_url,
            name="CARTO Voyager",
            attr=(
                '&copy; <a href="https://www.openstreetmap.org/copyright">'
                'OpenStreetMap</a> contributors '
                '&copy; <a href="https://carto.com/attributions">CARTO</a>'
            ),
            overlay=False,
            control=False,
        ).add_to(fire_map)
    else:
        folium.TileLayer(
            tiles="OpenStreetMap",
            name="OpenStreetMap",
            overlay=False,
            control=False,
        ).add_to(fire_map)

    # Official current fire perimeters.
    if not perimeter_data.empty:
        perimeter_layer = folium.FeatureGroup(
            name="NIFC current fire perimeters",
            show=True,
        )

        available_tooltip_fields = [
            field for field in [
                "poly_IncidentName",
                "poly_GISAcres",
                "attr_PercentContained",
                "attr_POOState",
            ]
            if field in perimeter_data.columns
        ]
        aliases = {
            "poly_IncidentName": "Fire:",
            "poly_GISAcres": "Mapped acres:",
            "attr_PercentContained": "Contained (%):",
            "attr_POOState": "State:",
        }

        geojson_options = {
            "data": perimeter_data.to_json(),
            "style_function": lambda feature: {
                "color": "#8B1E2D",
                "weight": 2.2,
                "opacity": 0.95,
                "fillColor": "#E76F51",
                "fillOpacity": 0.18,
            },
            "highlight_function": lambda feature: {
                "color": "#5B0B17",
                "weight": 3.5,
                "fillOpacity": 0.28,
            },
        }

        if available_tooltip_fields:
            geojson_options["tooltip"] = folium.GeoJsonTooltip(
                fields=available_tooltip_fields,
                aliases=[aliases[field] for field in available_tooltip_fields],
                localize=True,
                sticky=False,
            )

        folium.GeoJson(**geojson_options).add_to(perimeter_layer)
        perimeter_layer.add_to(fire_map)

    # One toggleable layer per FIRMS sensor.
    for source, style in FIRMS_SOURCES.items():
        detections = firms_layers.get(source)
        if detections is None or detections.empty:
            continue

        layer = folium.FeatureGroup(
            name=f"{style['label']} heat detections",
            show=style["show"],
        )

        for _, row in detections.iterrows():
            acq_time = clean_value(row.get("acq_time"))
            if acq_time != "N/A":
                acq_time = acq_time.zfill(4)

            popup = (
                f"<b>{escape(style['label'])} heat detection</b><br>"
                f"Date: {clean_value(row.get('acq_date'))}<br>"
                f"Time (UTC): {acq_time}<br>"
                f"FRP: {clean_value(row.get('frp'))} MW<br>"
                f"Confidence: {clean_value(row.get('confidence'))}"
            )

            folium.CircleMarker(
                location=[row["latitude"], row["longitude"]],
                radius=style["radius"],
                color=style["color"],
                weight=0.6,
                opacity=0.9,
                fill=True,
                fill_color=style["color"],
                fill_opacity=0.72,
                popup=folium.Popup(popup, max_width=280),
            ).add_to(layer)

        layer.add_to(fire_map)

    # Fit the initial view to the lower 48 and discourage panning far away.
    fire_map.fit_bounds(CONUS_BOUNDS, padding=(8, 8))
    fire_map.options["maxBounds"] = [[20.0, -132.0], [54.0, -60.0]]
    fire_map.options["maxBoundsViscosity"] = 0.75

    updated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    title_html = f''' 
    <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%);
                z-index: 9999; background: rgba(255,255,255,0.94); padding: 7px 12px;
                border: 1px solid #999; border-radius: 5px; font: 13px Arial;
                box-shadow: 0 1px 4px rgba(0,0,0,0.25);">
      <b>Current CONUS wildfire activity</b> &nbsp; Updated {updated}
    </div>
    '''
    fire_map.get_root().html.add_child(folium.Element(title_html))

    legend_rows = [
        '<div><span style="display:inline-block;width:18px;border-top:3px solid #8B1E2D;'
        'margin-right:7px;vertical-align:middle"></span>NIFC perimeter</div>'
    ]
    for style in FIRMS_SOURCES.values():
        legend_rows.append(
            f'<div><span style="display:inline-block;width:8px;height:8px;border-radius:50%;'
            f'background:{style["color"]};margin-right:7px"></span>{escape(style["label"])}</div>'
        )

    legend_html = f'''
    <div style="position: fixed; bottom: 25px; left: 10px; z-index: 9999;
                background: rgba(255,255,255,0.94); padding: 9px 11px;
                border: 1px solid #999; border-radius: 5px; font: 12px Arial;
                line-height: 1.65; box-shadow: 0 1px 4px rgba(0,0,0,0.2);">
      <b>Map layers</b><br>{''.join(legend_rows)}
    </div>
    '''
    fire_map.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl(collapsed=False).add_to(fire_map)
    return fire_map


fire_map = build_fire_map(firms_by_source, perimeters)
display(fire_map)

## 5. Save a standalone HTML map

The HTML file is a snapshot. It contains the data downloaded when this notebook ran.

In [ ]:
fire_map.save(OUTPUT_HTML)
print(f"Saved: {OUTPUT_HTML.resolve()}")

## 6. Optional automatic refresh

Set `AUTO_REFRESH = True` and run the cell below to download fresh data and replace the HTML every 10 minutes. Stop it with **Kernel → Interrupt**.

This works only while the Jupyter kernel remains running. For unattended 24/7 updates, schedule the notebook or a Python script on a server.

In [ ]:
AUTO_REFRESH = False
REFRESH_MINUTES = 10


def refresh_once():
    latest_firms = download_all_firms()
    latest_perimeters = download_current_perimeters()
    latest_map = build_fire_map(latest_firms, latest_perimeters)
    latest_map.save(OUTPUT_HTML)
    return latest_map


if AUTO_REFRESH:
    try:
        while True:
            refreshed_map = refresh_once()
            clear_output(wait=True)
            print(
                "Last refresh:",
                datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            )
            display(refreshed_map)
            time.sleep(REFRESH_MINUTES * 60)
    except KeyboardInterrupt:
        print("Automatic refresh stopped.")
else:
    print("Automatic refresh is off. Change AUTO_REFRESH to True to enable it.")

## Notes

- **Small dots do not mean small fires.** Each dot is a satellite heat detection.
- **MODIS is coarser than VIIRS,** so its marker is slightly larger and may overlap VIIRS detections.
- **Current WFIGS perimeters are not a complete history.** They are the current operational perimeter feed.
- This notebook intentionally leaves out the unfinished `ngfs_detections` cell from the original notebook. NOAA NGFS can be added later as a separate, properly downloaded layer.